# Testing and Evaluation

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from datetime import datetime

from src.evaluation_utils import calculate_fluency, calculate_readability_score, calculate_groundedness, calculate_relevance
from src.processing import clean_outputs, call_llm_async, read_write_data
from config.params import PARAMS

In [ ]:
# Logic needs updating if we have multiple runs with the same name
# Use current_time field to filter as well as run_name
run_name = ""

In [ ]:
patients_output = read_write_data("patients_output", "read")

# Read full table, then filter in pandas
journeys_and_notes = read_write_data("journeys", "read")
journeys_and_notes = journeys_and_notes[
    journeys_and_notes["run_name"] == run_name
]

admission_ids = set(journeys_and_notes["admission_id"])

# Read full table, then filter in pandas
admissions = read_write_data("admissions", "read")
admissions = admissions[
    admissions["admission_id"].isin(admission_ids)
]

## First, we will assess the fluency and readability.

- Readability socres are simple algorithms that rank how easy to read a string is. 
- Fluency uses an LLM as a Judge to measure how well a string flows and how easy it is to read.

In [ ]:
journeys_and_notes

In [ ]:
for score in ["flesch_reading_ease", "dale_chall_readability_score"]:
    journeys_and_notes[score] = journeys_and_notes['clean_note_text'].apply(lambda x: calculate_readability_score(x, score))

In [ ]:
notes = list(journeys_and_notes["clean_note_text"])[0:2]
fluency_scores = clean_outputs((await calculate_fluency(notes)), "dictionary")

journeys_and_notes["fluency_reasoning"] = [f["reasoning"] for f in fluency_scores]
journeys_and_notes["fluency_score"] = [f["score"] for f in fluency_scores]

## Next, we will look at groundedness and relevancy

- Groundedness uses an LLM as a Judge to measure how grounded the clinical note is, based on the information given in the pateint information and the event the note is referring to.
- Relevancy uses an LLM as a Judge to measure how relevant all the information is in the clincal note, and checks if any important information is missing.

In [ ]:
notes = list(journeys_and_notes["clean_note_text"])
events = list(journeys_and_notes["current_event_i"])
patients = [
    patients_output[patients_output["person_id"] == pid].to_dict('records') for pid in journeys_and_notes["patient_id"]
]

for judge in ["groundedness", "relevance"]:

    if judge == "groundedness":
        judge_response = clean_outputs(await calculate_groundedness(notes, events, patients), "dictionary")
    elif judge == "relevance":
        judge_response = clean_outputs(await calculate_relevance(notes, events, patients), "dictionary")

    journeys_and_notes[f"{judge}_reasoning"] = [f["reasoning"] for f in judge_response]
    journeys_and_notes[f"{judge}_score"] = [f["score"] for f in judge_response]

# Check time sequence between events

In [ ]:
def calculate_time_score(journeys_and_notes):
    """
    Checks if all events occur at the same time or after each other, for each patient.
    """
    # Get a list of all time events for each patient
    patients_times = journeys_and_notes.groupby("patient_id")["updt_dt_tm"].apply(list)
    
    ordered_events = []
    
    # For each pateint, add True of False to ordered_events depending on if all events occur equal to or after each other.
    for patient in patients_times:
        ordered_events.append(all([patient[i+1] >= patient[i] for i in range(len(patient)-1)]))
    
    # Return the fraction of patients where all events occur in time order.
    return ordered_events.count(True) / len(ordered_events)

In [ ]:
time_score = calculate_time_score(journeys_and_notes)

## Explore the Results

In [ ]:
for score, reasoning, note in zip(journeys_and_notes["groundedness_score"], journeys_and_notes["groundedness_reasoning"], journeys_and_notes["clean_note_text"]):
    if score == "3":
        print(score,"\n",reasoning,"\n", note)

In [ ]:
for score, reasoning in zip(journeys_and_notes["relevance_score"], journeys_and_notes["relevance_reasoning"]):
    print(score,"\n",reasoning,"\n")

## Save the Results

Fill in the below notes:

In [ ]:
evaluation_data = {"date": datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
                   "number_of_clinical_notes": len(journeys_and_notes),
                   "evaluation_notes": run_name,
                   "params": PARAMS,
                   "time_score" : time_score}

for col_name in ["flesch_reading_ease", "dale_chall_readability_score", "groundedness_score", "fluency_score", "relevance_score"]:
    col = np.array([float(n) for n in journeys_and_notes[col_name]]) 
    evaluation_data[col_name + "_min"] = np.min(col)
    evaluation_data[col_name + "_max"] = np.max(col)
    evaluation_data[col_name + "_mean"] = np.mean(col)

evaluation_table = pd.DataFrame([evaluation_data])

In [ ]:
evaluation_table

In [ ]:
try:
    
    #Read the evaluation dataset
    evaluation_results_df = read_write_data("evaluation_results", "read")
    #Join new results
    evaluation_concat = pd.concat([evaluation_results_df, evaluation_table])
    read_write_data("evaluation_results", "write", evaluation_concat)
    print("Appended results to evaluation_results")
except:
    print("Initialising evaluation_results table")
    read_write_data("evaluation_results", "write", evaluation_table)
    